# 30 CPU data sources, labels, and v2 reuse

CPU / network。Statcast・video source・v1 reuse・player-season BA/OPS/OBP/SLG labels・readiness をまとめます。

各 stage は `BASE_DIR/reports/preflight/compact_runs/` に JSONL log を追記します。既に期待 artifact が揃っている stage は skip します。再実行したい場合は `FORCE_RERUN_ALL=True` にしてください。

In [ ]:
from pathlib import Path
import importlib.util
import os
import sys


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


# v2 をデフォルトにします。v1 を再実行したい時だけここを書き換えてください。
os.environ.setdefault('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
os.environ.setdefault('BATTING_CODE_ROOT', '/content/drive/MyDrive/codex/batting_codex_handoff')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path(os.environ['BATTING_CODE_ROOT'])
    mydrive = Path('/content/drive/MyDrive')
    candidates = [
        fixed,
        Path('/content/drive/MyDrive/codex/batting_codex_handoff'),
        Path('/content/drive/MyDrive/batting_codex_handoff'),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            return candidate
    raise FileNotFoundError(
        'sport_pipeline repo が見つかりません。Drive mount と repo 配置を確認してください。\n'
        f'BATTING_CODE_ROOT={fixed}\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別配置の場合は compact notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/あなたの配置/batting_codex_handoff\n'
        f'MyDrive exists={mydrive.exists()}\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()
REPO_DIR = _resolve_repo_dir()
os.environ['BATTING_CODE_ROOT'] = str(REPO_DIR)
NOTEBOOK_DIR = REPO_DIR / 'notebooks'
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ['BASEBALL_VISION_RUN_PROFILE']
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME
src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(f'sport_pipeline を import できません: {src_dir}')

from sport_pipeline.compact_notebooks import CompactRunLogger
from sport_pipeline.pipeline.run_profile import artifact_id, load_run_profile, run_id

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')
CONTEXT_RUN_ID = run_id(RUN_PROFILE, 'context_run_id')
SEQUENCE_RUN_ID = run_id(RUN_PROFILE, 'sequence_run_id')
SEQUENCE_TCN_RUN_ID = run_id(RUN_PROFILE, 'sequence_tcn_run_id')
VIDEO_LIGHTWEIGHT_RUN_ID = run_id(RUN_PROFILE, 'video_lightweight_run_id')
VIDEO_FROZEN_RUN_ID = run_id(RUN_PROFILE, 'video_frozen_run_id')
VIDEO_FINETUNE_RUN_ID = run_id(RUN_PROFILE, 'video_finetune_run_id')
PLAYER_SEASON_RUN_ID = run_id(RUN_PROFILE, 'player_season_run_id')
VLM_RUN_ID = run_id(RUN_PROFILE, 'vlm_run_id')
FUSION_RUN_ID = run_id(RUN_PROFILE, 'fusion_run_id')
METHOD_EVALUATION_REPORT_ID = run_id(RUN_PROFILE, 'method_evaluation_report_id')
VIDEO_ABLATION_REPORT_ID = run_id(RUN_PROFILE, 'video_ablation_report_id')
STRUCTURED_SEQUENCE_FEATURE_ID = artifact_id(RUN_PROFILE, 'structured_sequence_feature_id')
CLIP_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'clip_embedding_feature_id')
PLAYER_SEASON_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'player_season_embedding_feature_id')
VIDEO_LIGHTWEIGHT_FEATURE_ID = artifact_id(RUN_PROFILE, 'video_lightweight_feature_id', 'video_lightweight_features_mlb_2024_2026_v2')
VIDEO_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'video_embedding_feature_id')
VLM_FEATURE_ID = artifact_id(RUN_PROFILE, 'vlm_feature_id')

print('RUN_PROFILE =', RUN_PROFILE_NAME)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('BASE_DIR =', BASE_DIR)
print('NOTEBOOK_DIR =', NOTEBOOK_DIR)
print('src_dir =', src_dir)


In [ ]:
COMPACT_RUN_NAME = '30_cpu_data_sources_labels_reuse'
LOGGER = CompactRunLogger(BASE_DIR, COMPACT_RUN_NAME, run_profile_name=RUN_PROFILE_NAME)
FORCE_RERUN_ALL = False

RUN_ENV_AND_INIT = True
RUN_RUN_ISOLATION_CHECK = True
RUN_SOURCE_REFRESH = False  # v1 で 11_download が済んでいるなら False のまま
RUN_V2_REUSE_FROM_V1 = True
RUN_PLAYER_SEASON_STATS = True
RUN_READINESS = True

stages = [
    {
        'notebook': '00_check_env.ipynb',
        'label': 'Environment check',
        'enabled': RUN_ENV_AND_INIT,
        'force': FORCE_RERUN_ALL,
        'expected_outputs': ['reports/preflight/check_env.json'],
        'progress_files': ['reports/preflight/check_env.json'],
    },
    {
        'notebook': '01_init_drive.ipynb',
        'label': 'Initialize Drive artifact directories',
        'enabled': RUN_ENV_AND_INIT,
        'force': FORCE_RERUN_ALL,
        'expected_outputs': ['reports/preflight/init_drive.json'],
        'progress_files': ['reports/preflight/init_drive.json'],
    },
    {
        'notebook': '10b_run_isolation_check.ipynb',
        'label': 'v1/v2 run isolation check',
        'enabled': RUN_RUN_ISOLATION_CHECK,
        'force': True,
    },
    {
        'notebook': '11_download_statcast_and_video_sources.ipynb',
        'label': 'Refresh Statcast and video source manifests',
        'enabled': RUN_SOURCE_REFRESH,
        'force': FORCE_RERUN_ALL,
        'expected_outputs': ['manifests/bbe_events_v1.parquet', 'manifests/video_sources_v1.parquet'],
        'progress_files': ['reports/preflight/statcast_download_progress_v1.json', 'reports/preflight/statsapi_video_resolve_progress_v1.json'],
    },
    {
        'notebook': '11_e_seed_v2_download_manifest_from_v1.ipynb',
        'label': 'Seed v2 download manifest from v1 videos',
        'enabled': RUN_V2_REUSE_FROM_V1,
        'force': FORCE_RERUN_ALL,
        'expected_outputs': [f'raw_videos/{FULL_RUN_ID}/download_manifest_v1.parquet'],
        'progress_files': [f'reports/preflight/reuse_download_manifest_{FULL_RUN_ID}.json'],
    },
    {
        'notebook': '11_f_download_player_season_batting_stats.ipynb',
        'label': 'Download player-season BA/OPS/OBP/SLG labels',
        'enabled': RUN_PLAYER_SEASON_STATS,
        'force': FORCE_RERUN_ALL,
        'expected_outputs': ['manifests/player_season_batting_v1.parquet'],
        'progress_files': ['reports/preflight/player_season_batting_stats_v1.json'],
    },
    {
        'notebook': '10_full_run_readiness.ipynb',
        'label': 'Full-run readiness check',
        'enabled': RUN_READINESS,
        'force': True,
        'expected_outputs': [f'reports/preflight/full_run_readiness_{FULL_RUN_ID}.json'],
        'progress_files': [f'reports/preflight/full_run_readiness_{FULL_RUN_ID}.json'],
    },
]
LOGGER.run_stages(ipython=get_ipython(), notebook_dir=NOTEBOOK_DIR, stages=stages)
